# Wintermute examples and testing pad

## Summary
This is the jupyter notebook to test all the different setups and to show people how it works.

## Modules we are testing
First we test the operation, pentesting, creating Devices, peripherals, users, analysts and all complete operation.

In [1]:
from wintermute.core import AWSAccount, Operation
from wintermute.findings import ReproductionStep, Risk, Vulnerability

# We create an operation

op = Operation("testOp")
op.addAnalyst("Alice", "aalice", "alice@example.com")
op.addDevice("host1", "192.168.1.10", "aa:bb:cc:dd:ee:ff", "Linux", "host1.local")
op.addUser(uid="jsmith", name="John Smith", email="john@example.com", teams=["Red"])
op.devices[0].addService(
    protocol="ipv4",
    app="nginx",
    portNumber=80,
    banner="nginx test",
    transport_layer="HTTP",
)
op.devices[0].services[0].addVulnerability(
    title="Weak TLS",
    description="TLS 1.0 enabled",
    cvss=5,
    risk={"likelihood": "High", "impact": "Medium", "severity": "Moderate"},
)

True

## Vulnerabilities, Reproduction Steps, and Risks

We can create vulnerabilities, which contain reproduction steps and each vulnerability contains a risk. This allows us to have reprodusible executions using either AI or the same functions for quick retesting and verification.

In [2]:
vuln = op.devices[0].services[0].vulnerabilities[0]
rs = ReproductionStep(
    "Step 1",
    "This is the first step for reproduction",
    "gcc",
    "compile",
    5,
    ["-o", "xploit", "xploit.c"],
)
vuln.reproduction_steps.append(rs)
rs2 = ReproductionStep(
    "Step 2",
    "This is the second step for reproduction",
    "./exploit",
    "compile",
    5,
    [],
    "sh#",
)
vuln.reproduction_steps.append(rs2)

# AWS Account
acct = AWSAccount(
    accountId=1234567890,
    name="aws-prod",
    vulnerabilities=[
        Vulnerability(
            title="S3 bucket public",
            description="Bucket allows public read",
            risk=Risk(likelihood="High", impact="Medium", severity="High"),
            reproduction_steps=[
                ReproductionStep(
                    title="List objects",
                    tool="aws",
                    action="s3 ls",
                    arguments=["s3://bucket"],
                )
            ],
            verified=True,
        )
    ],
)
op.awsaccounts.append(acct)

## Backends (Tickets, AI, utils, etc)

Wintermute has a backend setup that supports multiple types of backends. For now it holds:

- Tickets (Bugzilla, Memory)
- AI (Bedrock aka AWS, openAI, Grok)
- Depthcharge (Secure boot attack framework)
- Docx Reports (Using templates)

### Report() 
Report is a metaclass, that you can use regardless of the backend, meaning if you want to change the backend from docx, to any custom HTML, Markdown, Ticketing System, Wiki, the script calling Report will need no changes just changing the backend directly should make it work without any problems and transparently. This allows for easier cartridge (aka high level automation) creation without having to make hard changes with new backends or new technologies added.

In [12]:
from wintermute.backends.docx_reports import DocxTplPerVulnBackend
from wintermute.reports import Report, ReportSpec

# We register the word backend as the Report (metaclass) backend agent
Report.register_backend(
    "word_tpl_per_vuln",
    DocxTplPerVulnBackend(
        template_dir="templates",
        main_template="report_main.docx",
        vuln_template="report_vuln.docx",
    ),
    make_default=True,
)

# We create the report specs (Title, author, summary, etc)
spec = ReportSpec(
    title="Security Assessment Example",
    author="Enrique Sanchez",
    summary="We should make this actually do it with AI..",
)

# We save the report, this in case it was for a wiki, Jira, etc will do the full execution
Report.save(spec, [op], "out.docx")

# Print operation
# import pprint
# pprint.pprint(op.to_dict())

### Ticket()
The Ticket() metaclass allows you to mantain that part of the automation unchanged while just changing the backend. The following is an example of the Ticket usage using the "in-memory" ticket backend engine first.

In [5]:
from wintermute.tickets import InMemoryBackend, Status, Ticket

Ticket.register_backend("mem", InMemoryBackend(), make_default=True)
tid = Ticket.create(
    title="Login bug example", description="Fails on Safari for some reason"
)
Ticket.comment(tid, text="Reproduced on 17.0.3", author="qa-bot")
t = Ticket.read(tid)
print(t.to_dict())
print(f"Current ticket status: {t.to_dict().get('data').get('status')}")
print(f"Changing ticket {tid} status to 'IN_PROGRESS'")
Ticket.update(tid, status=Status.IN_PROGRESS)
print(f"Current ticket status: {t.to_dict().get('data').get('status')}")
print("Changing requester field to 'foobar'")
Ticket.update(tid, requester="foobar")
print(t.to_dict())

{'ticket_id': 'T000001', 'data': {'title': 'Login bug example', 'description': 'Fails on Safari for some reason', 'assignee': None, 'requester': None, 'status': 'open', 'custom_fields': {}, 'communication': []}, 'comments': [{'author': 'qa-bot', 'text': 'Reproduced on 17.0.3', 'at': '2025-11-28T05:20:31.601513Z'}]}
Current ticket status: open
Changing ticket T000001 status to 'IN_PROGRESS'
Current ticket status: in_progress
Changing requester field to 'foobar'
{'ticket_id': 'T000001', 'data': {'title': 'Login bug example', 'description': 'Fails on Safari for some reason', 'assignee': None, 'requester': 'foobar', 'status': 'in_progress', 'custom_fields': {}, 'communication': []}, 'comments': [{'author': 'qa-bot', 'text': 'Reproduced on 17.0.3', 'at': '2025-11-28T05:20:31.601513Z'}]}
